In [44]:

from python_utilities.db_connection import DbConnection
import boto3
import json
import os
import pandas as pd
import ast

analytics_db = DbConnection('PROD', 'PROD_DE')
# Create session with specific profile
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

INFO:root:PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [2]:
new_1 = pd.read_csv('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/letters/courteam_backlog_letters/data/pf-ger-legal-after-court-gerichtsvollzieher-view-2026-07-09-1335_NEW1.csv')
new_2 = pd.read_csv('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/letters/courteam_backlog_letters/data/pf-ger-legal-after-court-verm_gensverzeichnisdrittauskunft-view-2026-07-09-1151_NEW2.csv')

In [3]:
new_data = pd.concat([new_1, new_2], ignore_index=True)

In [4]:
new_data

,ID,Status,Case ID,Zufriedenheit,Betreff,Anfragender,Angefragt,Mitarbeiter
0,19072799,Neu,195479465492,Nicht angeboten,"Ihr Zeichen: 195479465492, Mein Zeichen: DR II...",Gerichtsvollzieher Jödicke,2026-07-09 13:29,NaN
1,19072672,Neu,170271494966,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
2,19072671,Neu,110641722258,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
3,19072667,Neu,180017641051,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
4,19072659,Neu,168877430715,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
...,...,...,...,...,...,...,...,...
6033,19068087,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:26,NaN
6034,19068095,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:27,NaN
6035,19068207,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:35,NaN
6036,19069950,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 10:18,NaN


In [5]:
data_one = pd.read_csv('data/pf-ger-legal-after-court-1st-level-after-court-view-2026-06-18-1610.csv')
data_two = pd.read_csv('data/pf-ger-legal-after-court-gerichtsvollzieher-view-2026-06-18-1243.csv')
data_three = pd.read_csv('data/pf-ger-legal-after-court-verm_gensverzeichnisdrittauskunft-view-2026-06-18-1610.csv')

In [6]:
# print(data_one.shape, data_two.shape, data_three.shape)

# for df in [data_one, data_two, data_three]:
#     print(df.columns)

In [7]:
# data_two

In [8]:
# data_three.head()

In [9]:
cols_1 = set(data_one.columns)
cols_2 = set(data_two.columns)
cols_3 = set(data_three.columns)
cols_4 = set(new_data.columns)


common_cols = cols_1.intersection(cols_2).intersection(cols_3).intersection(cols_4)
print(common_cols)

{'Status', 'Betreff', 'Mitarbeiter', 'Angefragt', 'Anfragender', 'ID'}


In [10]:
data_one =  data_one[list(common_cols)]
data_two =  data_two[list(common_cols)]
data_three =  data_three[list(common_cols)]
all_data = pd.concat([data_one, data_two, data_three], ignore_index=True)
all_data.rename(columns={'ID': 'zendesk_id'}, inplace=True)

In [11]:
all_data

,Status,Betreff,Mitarbeiter,Angefragt,Anfragender,zendesk_id
0,Neu,1st Level (in after-court area),NaN,2026-06-18 10:02,PairFinance,18879333
1,Neu,1st Level (in after-court area),NaN,2026-06-18 09:49,PairFinance,18879149
2,Neu,1st Level (in after-court area),NaN,2026-06-18 09:43,PairFinance,18879079
3,Neu,1st Level (in after-court area),NaN,2026-06-18 09:43,PairFinance,18879069
4,Neu,1st Level (in after-court area),NaN,2026-06-18 09:24,PairFinance,18878724
...,...,...,...,...,...,...
8491,Neu,Vermögensverzeichnis/Drittauskunft,NaN,2026-06-18 10:12,PairFinance,18879645
8492,Neu,Vermögensverzeichnis/Drittauskunft,NaN,2026-06-18 10:12,PairFinance,18879649
8493,Neu,Vermögensverzeichnis/Drittauskunft,NaN,2026-06-18 10:13,PairFinance,18879655
8494,Neu,Vermögensverzeichnis/Drittauskunft,NaN,2026-06-18 10:13,PairFinance,18879667


In [12]:
new_data.rename(columns={'ID': 'zendesk_id'}, inplace=True)

In [13]:
new_data_to_run = new_data[~new_data['zendesk_id'].isin(all_data['zendesk_id'])]

In [14]:
new_data_to_run

,zendesk_id,Status,Case ID,Zufriedenheit,Betreff,Anfragender,Angefragt,Mitarbeiter
0,19072799,Neu,195479465492,Nicht angeboten,"Ihr Zeichen: 195479465492, Mein Zeichen: DR II...",Gerichtsvollzieher Jödicke,2026-07-09 13:29,NaN
1,19072672,Neu,170271494966,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
2,19072671,Neu,110641722258,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
3,19072667,Neu,180017641051,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
4,19072659,Neu,168877430715,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN
...,...,...,...,...,...,...,...,...
6033,19068087,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:26,NaN
6034,19068095,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:27,NaN
6035,19068207,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:35,NaN
6036,19069950,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 10:18,NaN


In [15]:
all_data.Status.value_counts()

Status
Neu        6958
Offen      1524
Wartend      14
Name: count, dtype: int64

In [16]:
new_data_to_run.Status.value_counts()

Status
Neu        4039
Offen        37
Wartend       3
Name: count, dtype: int64

In [17]:
# query = """SELECT *
# FROM public.raw_zendesk_tickets
# WHERE channel = 'letter'
# and attachments != '[]'
# LIMIT 5000"""
# data = analytics_db.sql_to_df(query)
# sample = data.sample(1000, random_state=42)

In [18]:
# fetch_zendesk_ids = all_data['zendesk_id'].tolist()
# fetch_zendesk_ids_str = ','.join("'" + str(i) + "'" for i in fetch_zendesk_ids)

In [19]:
fetch_zendesk_ids = new_data_to_run['zendesk_id'].tolist()
fetch_zendesk_ids_str = ','.join("'" + str(i) + "'" for i in fetch_zendesk_ids)

In [20]:

query = f"""SELECT rzc.zendesk_ticket_id, rzc.zendesk_comment_id, rzc.attachments, rzc.created_at, rzt.channel, rzc.body, rzt.content
FROM public.raw_zendesk_tickets rzt
JOIN public.raw_zendesk_comments rzc ON rzt.zendesk_ticket_id = rzc.zendesk_ticket_id
WHERE rzt.zendesk_ticket_id IN ({fetch_zendesk_ids_str})
"""

prod_data = analytics_db.sql_to_df(query)

In [21]:
prod_data.attachments.isna().sum()

np.int64(0)

In [22]:
prod_data

,zendesk_ticket_id,zendesk_comment_id,attachments,created_at,channel,body,content
0,18540147,27372456043548,"[{'id': '27372456043548-1', 'url': 'https://su...",2026-05-11 04:43:36.648190,letter,Detected Slugs:,Vermögensverzeichnis/Drittauskunft\nDetected S...
1,18899849,28334042955420,"[{'id': '28334042955420-1', 'url': 'https://su...",2026-06-21 11:18:13.783781,letter,Detected Slugs:\n\n 131345401171,Vermögensverzeichnis/Drittauskunft\nDetected S...
2,18894929,28313637251868,"[{'id': '28313637251868-1', 'url': 'https://su...",2026-06-20 03:46:46.500877,letter,Detected Slugs:\n\n 156190292045\n 30074...,GVZ - Scanned Brief\nDetected Slugs:\n\n 15...
3,18894943,28313651595292,"[{'id': '28313651595292-1', 'url': 'https://su...",2026-06-20 03:47:02.704432,letter,Detected Slugs:\n\n 128031069569\n 84300...,GVZ - Scanned Brief\nDetected Slugs:\n\n 12...
4,18894959,28313669436060,"[{'id': '28313669436060-1', 'url': 'https://su...",2026-06-20 03:47:35.604214,letter,Detected Slugs:\n\n 678605559210901\n 18...,GVZ - Scanned Brief\nDetected Slugs:\n\n 67...
...,...,...,...,...,...,...,...
4080,19072650,28787251062044,"[{'id': '28787251062044-1', 'url': 'https://su...",2026-07-09 11:18:18.455182,letter,Detected Slugs:\n\n 198171780044,GVZ - Scanned Brief\nDetected Slugs:\n\n 19...
4081,19072653,28787266448412,"[{'id': '28787266448412-1', 'url': 'https://su...",2026-07-09 11:18:23.305138,letter,Detected Slugs:\n\n 178879721508,GVZ - Scanned Brief\nDetected Slugs:\n\n 17...
4082,19072598,28787173319708,"[{'id': '28787173319708-1', 'url': 'https://su...",2026-07-09 11:14:51.944369,email,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ihr Zeichen: 159111921478, Mein Zeichen: DR II..."
4083,19072671,28787282083740,"[{'id': '28787282083740-1', 'url': 'https://su...",2026-07-09 11:18:52.416162,letter,Detected Slugs:\n\n 110641722258,GVZ - Scanned Brief\nDetected Slugs:\n\n 11...


In [23]:
# check = prod_data.attachments == '[]'
# print(check.sum())

In [24]:
# prod_data

In [25]:
# # merged two data
new_data_to_run['zendesk_id'] = new_data_to_run['zendesk_id'].astype(str)
prod_data['zendesk_ticket_id'] = prod_data['zendesk_ticket_id'].astype(str)
final_data_new_run = pd.merge(new_data_to_run, prod_data, left_on='zendesk_id', right_on='zendesk_ticket_id', how='left')

/var/folders/ym/hcyz4chn3cq4n_8dslg5k50h0000gn/T/ipykernel_17892/249884806.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data_to_run['zendesk_id'] = new_data_to_run['zendesk_id'].astype(str)


In [27]:
final_data_new_run

,zendesk_id,Status,Case ID,Zufriedenheit,Betreff,Anfragender,Angefragt,Mitarbeiter,zendesk_ticket_id,zendesk_comment_id,attachments,created_at,channel,body,content
0,19072799,Neu,195479465492,Nicht angeboten,"Ihr Zeichen: 195479465492, Mein Zeichen: DR II...",Gerichtsvollzieher Jödicke,2026-07-09 13:29,NaN,19072799,28787575304348,"[{'id': '28787575304348-1', 'url': 'https://su...",2026-07-09 11:29:48.315570,email,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ihr Zeichen: 195479465492, Mein Zeichen: DR II..."
1,19072672,Neu,170271494966,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN,19072672,28787295872668,"[{'id': '28787295872668-1', 'url': 'https://su...",2026-07-09 11:18:57.264816,letter,Detected Slugs:\n\n 170271494966,GVZ - Scanned Brief\nDetected Slugs:\n\n 17...
2,19072671,Neu,110641722258,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN,19072671,28787282083740,"[{'id': '28787282083740-1', 'url': 'https://su...",2026-07-09 11:18:52.416162,letter,Detected Slugs:\n\n 110641722258,GVZ - Scanned Brief\nDetected Slugs:\n\n 11...
3,19072667,Neu,180017641051,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN,19072667,28787275402652,"[{'id': '28787275402652-1', 'url': 'https://su...",2026-07-09 11:18:50.862713,letter,Detected Slugs:\n\n 180017641051,GVZ - Scanned Brief\nDetected Slugs:\n\n 18...
4,19072659,Neu,168877430715,Nicht angeboten,GVZ - Scanned Brief,PairFinance,2026-07-09 13:18,NaN,19072659,28787280433564,"[{'id': '28787280433564-1', 'url': 'https://su...",2026-07-09 11:18:33.410811,letter,Detected Slugs:\n\n 168877430715,GVZ - Scanned Brief\nDetected Slugs:\n\n 16...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4080,19068087,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:26,NaN,19068087,28779319216284,"[{'id': '28779319216284-1', 'url': 'https://su...",2026-07-09 06:26:36.094517,letter,Detected Slugs:\n\n 194565595460,Vermögensverzeichnis/Drittauskunft\nDetected S...
4081,19068095,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:27,NaN,19068095,28779327609372,"[{'id': '28779327609372-1', 'url': 'https://su...",2026-07-09 06:27:14.520869,letter,Detected Slugs:\n\n 130564406812,Vermögensverzeichnis/Drittauskunft\nDetected S...
4082,19068207,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 08:35,NaN,19068207,28779462412572,"[{'id': '28779462412572-1', 'url': 'https://su...",2026-07-09 06:35:29.195096,letter,Detected Slugs:\n\n 158735276640,Vermögensverzeichnis/Drittauskunft\nDetected S...
4083,19069950,Neu,NaN,NaN,Vermögensverzeichnis/Drittauskunft,PairFinance,2026-07-09 10:18,NaN,19069950,28782152944796,"[{'id': '28782152944796-1', 'url': 'https://su...",2026-07-09 08:18:40.410221,letter,Detected Slugs:\n\n 152648433026,Vermögensverzeichnis/Drittauskunft\nDetected S...


In [28]:
# final_data.channel.value_counts()

In [29]:
# import json
# import os
# import pandas as pd
# import ast

# # Load download cache to get attachment IDs
# download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/court_team_backlog"
# # created_at_start = "2026-06-16"
# # created_at_end = "2026-06-17"
# # cache_file = os.path.join(download_dir, "_download_cache.json")

# # with open(cache_file, "r") as f:
# #     cache = json.load(f)

# # attachment_ids = list(cache["downloaded"].keys())
# # print(f"Found {len(attachment_ids)} attachment IDs in download cache")

# # # Query DB for tickets that have these attachment IDs
# # ids_in_clause = ",".join("'" + aid + "'" for aid in attachment_ids)

# # query = f"""SELECT rzc.zendesk_ticket_id, rzc.zendesk_comment_id, rzc.attachments, rzc.created_at, rzt.channel, rzc.body, rzt.content
# # FROM public.raw_zendesk_tickets rzt
# # JOIN public.raw_zendesk_comments rzc ON rzt.zendesk_ticket_id = rzc.zendesk_ticket_id
# # WHERE rzt.channel = 'letter'
# # AND rzc.created_at >= '{created_at_start}'
# # AND rzc.created_at < '{created_at_end}'
# # AND rzc.attachments != '[]'
# # AND EXISTS (
# #     SELECT 1 FROM jsonb_array_elements(attachments::jsonb) elem
# #     WHERE (elem->>'id')::text IN ({ids_in_clause})
# # )"""

# # data = analytics_db.sql_to_df(query)
# print(f"Got {len(final_data)} tickets from DB")

# # Rebuild new_letter_df from these tickets, filtering to only cached attachment IDs
# rows = []

# for index, row in final_data.iterrows():
#     zendesk_id = row['zendesk_ticket_id']
#     comment_id = row['zendesk_comment_id']
#     created_at = row['created_at']
#     body = row['body']
#     content = row['content']
#     attachments = row['attachments']
#     if attachments is None:
#         continue
#     if isinstance(attachments, str):
#         if attachments == '[]':
#             continue
#         attachments = ast.literal_eval(attachments)
#     elif not isinstance(attachments, (list, tuple)):
#         if pd.isna(attachments):
#             continue
#     for attachment in attachments:
#         attachment_id = str(attachment['id'])
#         attachment_url = attachment['url']
#         rows.append({
#             'zendesk_id': zendesk_id,
#             'comment_id': comment_id,
#             'created_at': created_at,
#             'body': body,
#             'content': content,
#             'attachment_id': attachment_id,
#             'attachment_url': attachment_url
#         })

# backlog_df = pd.DataFrame(rows, columns=['zendesk_id', 'comment_id', 'created_at', 'body', 'content', 'attachment_id', 'attachment_url'])

In [30]:
# Rebuild new_letter_df from these tickets, filtering to only cached attachment IDs
rows = []

for index, row in final_data_new_run.iterrows():
    zendesk_id = row['zendesk_ticket_id']
    comment_id = row['zendesk_comment_id']
    created_at = row['created_at']
    channel = row['channel']
    body = row['body']
    content = row['content']
    attachments = row['attachments']
    if attachments is None:
        continue
    if isinstance(attachments, str):
        if attachments == '[]':
            continue
        attachments = ast.literal_eval(attachments)
    elif not isinstance(attachments, (list, tuple)):
        if pd.isna(attachments):
            continue
    for attachment in attachments:
        attachment_id = str(attachment['id'])
        attachment_url = attachment['url']
        rows.append({
            'zendesk_id': zendesk_id,
            'comment_id': comment_id,
            'created_at': created_at,
            'channel': channel,
            'body': body,
            'content': content,
            'attachment_id': attachment_id,
            'attachment_url': attachment_url
        })

backlog_df_new = pd.DataFrame(rows, columns=['zendesk_id', 'comment_id', 'created_at', 'channel', 'body', 'content', 'attachment_id', 'attachment_url'])

In [31]:
backlog_df_new.channel.value_counts()

channel
letter    3885
email      237
Name: count, dtype: int64

In [32]:
# use this below for textract
backlog_df_new.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/letters/courteam_backlog_letters/data/backlog_df_new.csv", index=False)

In [33]:
backlog_df_new = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/letters/courteam_backlog_letters/data/backlog_df_new.csv")

In [34]:
backlog_df_new

,zendesk_id,comment_id,created_at,channel,body,content,attachment_id,attachment_url
0,19072799,28787575304348,2026-07-09 11:29:48.315570,email,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ihr Zeichen: 195479465492, Mein Zeichen: DR II...",28787575304348-1,https://support.pairfinance.de/attachments/tok...
1,19072672,28787295872668,2026-07-09 11:18:57.264816,letter,Detected Slugs:\n\n 170271494966,GVZ - Scanned Brief\nDetected Slugs:\n\n 17...,28787295872668-1,https://support.pairfinance.de/attachments/tok...
2,19072671,28787282083740,2026-07-09 11:18:52.416162,letter,Detected Slugs:\n\n 110641722258,GVZ - Scanned Brief\nDetected Slugs:\n\n 11...,28787282083740-1,https://support.pairfinance.de/attachments/tok...
3,19072667,28787275402652,2026-07-09 11:18:50.862713,letter,Detected Slugs:\n\n 180017641051,GVZ - Scanned Brief\nDetected Slugs:\n\n 18...,28787275402652-1,https://support.pairfinance.de/attachments/tok...
4,19072659,28787280433564,2026-07-09 11:18:33.410811,letter,Detected Slugs:\n\n 168877430715,GVZ - Scanned Brief\nDetected Slugs:\n\n 16...,28787280433564-1,https://support.pairfinance.de/attachments/tok...
...,...,...,...,...,...,...,...,...
4117,19068087,28779319216284,2026-07-09 06:26:36.094517,letter,Detected Slugs:\n\n 194565595460,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779319216284-1,https://support.pairfinance.de/attachments/tok...
4118,19068095,28779327609372,2026-07-09 06:27:14.520869,letter,Detected Slugs:\n\n 130564406812,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779327609372-1,https://support.pairfinance.de/attachments/tok...
4119,19068207,28779462412572,2026-07-09 06:35:29.195096,letter,Detected Slugs:\n\n 158735276640,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779462412572-1,https://support.pairfinance.de/attachments/tok...
4120,19069950,28782152944796,2026-07-09 08:18:40.410221,letter,Detected Slugs:\n\n 152648433026,Vermögensverzeichnis/Drittauskunft\nDetected S...,28782152944796-1,https://support.pairfinance.de/attachments/tok...


In [35]:
backlog_df_new.content.value_counts()

content
Vermögensverzeichnis/Drittauskunft\nDetected Slugs:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [36]:
backlog_df_new.zendesk_id.is_unique

False

In [37]:
backlog_df_new.attachment_id.is_unique

True

In [38]:
backlog_df_new.comment_id.is_unique

False

In [39]:
backlog_df_new.shape

(4122, 8)

In [40]:
backlog_df_new.shape

(4122, 8)

In [41]:
import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')

In [42]:
# Inspect Textract cache size and how many attachments still need OCR
# (mirrors parse_url_attachments_with_textract: cache is keyed by S3 object key,
#  matched back to attachment_id via the key's basename).
import os
import json

cache_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/court_team_backlog"
textract_cache_file = os.path.join(cache_dir, "_textract_cache.json")

if os.path.exists(textract_cache_file):
    with open(textract_cache_file, "r") as f:
        textract_cache = json.load(f)
else:
    textract_cache = {}

# attachment_ids already present in the cache (basename without extension)
cached_attachment_ids = {
    os.path.splitext(os.path.basename(key))[0] for key in textract_cache
}

attachment_ids = backlog_df_new["attachment_id"].astype(str)
total = len(attachment_ids)
already_cached = attachment_ids.isin(cached_attachment_ids).sum()
remaining = total - already_cached

print(f"Textract cache file: {textract_cache_file}")
print(f"Cache entries (total cached docs): {len(textract_cache)}")
print(f"Attachments to process:            {total}")
print(f"  already cached (will skip OCR):  {already_cached}")
print(f"  remaining (will be OCR'd):       {remaining}")
print(f"Progress: {already_cached / total * 100:.1f}% cached" if total else "No attachments")


Textract cache file: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/court_team_backlog/_textract_cache.json
Cache entries (total cached docs): 8449
Attachments to process:            4122
  already cached (will skip OCR):  0
  remaining (will be OCR'd):       4122
Progress: 0.0% cached


In [45]:
import os
from utils.use_textract_utils import parse_url_attachments_with_textract
import logging
logging.basicConfig(level=logging.INFO, force=True)


cache_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/court_team_backlog"
os.makedirs(cache_dir, exist_ok=True)
textract_cache_file = os.path.join(cache_dir, "_textract_cache.json")

# Stream attachments straight from their token URLs into S3 (no local disk),
# then run Textract. S3 acts as the upload cache (skip_if_exists) and
# Textract results are cached to disk so re-runs don't re-OCR.
attachments = list(zip(backlog_df_new['attachment_id'].astype(str), backlog_df_new['attachment_url']))

results = parse_url_attachments_with_textract(
    attachments=attachments,
    s3_object_key_base="ocr_source_files/letters",
    use_page_markers=True,
    skip_if_exists=True,
    skip_upload=False,
    cache_path=textract_cache_file,
)

print(f"Parsed {len(results)} attachments with Textract")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:utils.use_textract_utils:[Textract Cache] Loaded 8449 cached entries
INFO:utils.use_textract_utils:[Upload] Starting upload phase for 4122 attachments
INFO:utils.use_textract_utils:[Upload] Skipped 'ocr_source_files/letters/28787575304348-1.pdf' (already in S3; 1 skipped so far)
INFO:utils.use_textract_utils:[Upload] Skipped 'ocr_source_files/letters/28787295872668-1.pdf' (already in S3; 2 skipped so far)
INFO:utils.use_textract_utils:[Upload] Skipped 'ocr_source_files/letters/28787282083740-1.pdf' (already in S3; 3 skipped so far)
INFO:utils.use_textract_utils:[Upload] Skipped 'ocr_source_files/letters/28787275402652-1.pdf' (already in S3; 4 skipped so far)
INFO:utils.use_textract_utils:[Upload] Skipped 'ocr_source_files/letters/28787280433564-1.pdf' (already in S3; 5 skipped so far)
INFO:utils.use_textract_utils:[Upload] Skipped 'ocr_source_files/letters/28787280233372-1.pdf' (already in S

KeyboardInterrupt: 

We are at 80%, lets run our app with them. we will run the rest later.

Current situation: attachments in backlog_df_new uploaded to the s3. I want to insert this data to llm attachments and llm tickets table, such that
it could be processed by our files. For that we need to add some columns to dataframe

In [47]:
backlog_df_new

,zendesk_id,comment_id,created_at,channel,body,content,attachment_id,attachment_url
0,19072799,28787575304348,2026-07-09 11:29:48.315570,email,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ihr Zeichen: 195479465492, Mein Zeichen: DR II...",28787575304348-1,https://support.pairfinance.de/attachments/tok...
1,19072672,28787295872668,2026-07-09 11:18:57.264816,letter,Detected Slugs:\n\n 170271494966,GVZ - Scanned Brief\nDetected Slugs:\n\n 17...,28787295872668-1,https://support.pairfinance.de/attachments/tok...
2,19072671,28787282083740,2026-07-09 11:18:52.416162,letter,Detected Slugs:\n\n 110641722258,GVZ - Scanned Brief\nDetected Slugs:\n\n 11...,28787282083740-1,https://support.pairfinance.de/attachments/tok...
3,19072667,28787275402652,2026-07-09 11:18:50.862713,letter,Detected Slugs:\n\n 180017641051,GVZ - Scanned Brief\nDetected Slugs:\n\n 18...,28787275402652-1,https://support.pairfinance.de/attachments/tok...
4,19072659,28787280433564,2026-07-09 11:18:33.410811,letter,Detected Slugs:\n\n 168877430715,GVZ - Scanned Brief\nDetected Slugs:\n\n 16...,28787280433564-1,https://support.pairfinance.de/attachments/tok...
...,...,...,...,...,...,...,...,...
4117,19068087,28779319216284,2026-07-09 06:26:36.094517,letter,Detected Slugs:\n\n 194565595460,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779319216284-1,https://support.pairfinance.de/attachments/tok...
4118,19068095,28779327609372,2026-07-09 06:27:14.520869,letter,Detected Slugs:\n\n 130564406812,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779327609372-1,https://support.pairfinance.de/attachments/tok...
4119,19068207,28779462412572,2026-07-09 06:35:29.195096,letter,Detected Slugs:\n\n 158735276640,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779462412572-1,https://support.pairfinance.de/attachments/tok...
4120,19069950,28782152944796,2026-07-09 08:18:40.410221,letter,Detected Slugs:\n\n 152648433026,Vermögensverzeichnis/Drittauskunft\nDetected S...,28782152944796-1,https://support.pairfinance.de/attachments/tok...


In [48]:
# 1) set created at to now, so it will be fetched by the pipeline.
backlog_df_new['created_at'] = pd.Timestamp.now()

In [49]:
# required columns 
# llm ticket
# ticket_uuid = auto create
# source_type = 'zendesk'
# egvp_id = none
# status = 'new'
# origin = 'DE'
# llm attachments
# status = 'new'
# file_name = "extract from attachment url"
# file_extension = "extract from attachment url"
# created_at = llm ticket created at
# s3_key = "ocr_source_files/letters/{attachment_id}.pdf"
# s3_bucket = "set bucket"

In [50]:
llm_ticket_insert = backlog_df_new.copy()
llm_attachment_insert = backlog_df_new.copy()

In [51]:
llm_ticket_insert

,zendesk_id,comment_id,created_at,channel,body,content,attachment_id,attachment_url
0,19072799,28787575304348,2026-07-15 12:04:28.618754,email,"Sehr geehrte Damen und Herren,\n\nSie erhalten...","Ihr Zeichen: 195479465492, Mein Zeichen: DR II...",28787575304348-1,https://support.pairfinance.de/attachments/tok...
1,19072672,28787295872668,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 170271494966,GVZ - Scanned Brief\nDetected Slugs:\n\n 17...,28787295872668-1,https://support.pairfinance.de/attachments/tok...
2,19072671,28787282083740,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 110641722258,GVZ - Scanned Brief\nDetected Slugs:\n\n 11...,28787282083740-1,https://support.pairfinance.de/attachments/tok...
3,19072667,28787275402652,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 180017641051,GVZ - Scanned Brief\nDetected Slugs:\n\n 18...,28787275402652-1,https://support.pairfinance.de/attachments/tok...
4,19072659,28787280433564,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 168877430715,GVZ - Scanned Brief\nDetected Slugs:\n\n 16...,28787280433564-1,https://support.pairfinance.de/attachments/tok...
...,...,...,...,...,...,...,...,...
4117,19068087,28779319216284,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 194565595460,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779319216284-1,https://support.pairfinance.de/attachments/tok...
4118,19068095,28779327609372,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 130564406812,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779327609372-1,https://support.pairfinance.de/attachments/tok...
4119,19068207,28779462412572,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 158735276640,Vermögensverzeichnis/Drittauskunft\nDetected S...,28779462412572-1,https://support.pairfinance.de/attachments/tok...
4120,19069950,28782152944796,2026-07-15 12:04:28.618754,letter,Detected Slugs:\n\n 152648433026,Vermögensverzeichnis/Drittauskunft\nDetected S...,28782152944796-1,https://support.pairfinance.de/attachments/tok...


In [52]:
import uuid
def generate_ticket_uuid(zendesk_id, comment_id, egvp_id):
    """
    Generate a deterministic UUID based on zendesk_id, comment_id, and egvp_id.
    Same triplet will always produce the same UUID.
    """
    # Create a stable string representation, handling nulls
    parts = [
        str(int(zendesk_id)) if pd.notna(zendesk_id) else "NULL",
        str(int(comment_id)) if pd.notna(comment_id) else "NULL",
        str(egvp_id) if pd.notna(egvp_id) else "NULL",
    ]
    name_string = "|".join(parts)

    # Use uuid5 with a custom namespace for deterministic UUID generation
    namespace = uuid.NAMESPACE_DNS  # DNS namespace
    return str(uuid.uuid5(namespace, name_string))


In [53]:
llm_ticket_insert["egvp_id"] = None

llm_ticket_insert['created_at'] = pd.Timestamp.now()

llm_ticket_insert["ticket_uuid"] = llm_ticket_insert.apply(
    lambda row: generate_ticket_uuid(
        row.get("zendesk_id"), row.get("comment_id"), row.get("egvp_id")
    ),
    axis=1,
)

llm_ticket_insert["source_type"] = "zendesk"
llm_ticket_insert["status"] = "new"
llm_ticket_insert["origin"] = "DE"
llm_ticket_insert["status_written_at"] = llm_ticket_insert["created_at"]


In [54]:
llm_ticket_insert_final = llm_ticket_insert[["ticket_uuid", "source_type", "zendesk_id", "comment_id", "egvp_id", "status", "origin", "created_at", "status_written_at"]]

In [55]:
llm_ticket_insert_final

,ticket_uuid,source_type,zendesk_id,comment_id,egvp_id,status,origin,created_at,status_written_at
0,75fc9480-1a6d-52b1-a8ba-d861655f9b27,zendesk,19072799,28787575304348,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
1,40033f7e-9347-5c15-a9fc-a29db526a08d,zendesk,19072672,28787295872668,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
2,8a7ef91a-394f-5d13-987c-3e3cd8cbcb38,zendesk,19072671,28787282083740,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
3,5a9dfa43-6bfa-50e1-a790-7a33647405f2,zendesk,19072667,28787275402652,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4,ce30ef5a-b0f1-518e-a796-214519da89bf,zendesk,19072659,28787280433564,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
...,...,...,...,...,...,...,...,...,...
4117,f33eebda-16a8-51f7-839f-d8482f3e93ad,zendesk,19068087,28779319216284,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4118,694e0ba9-08c4-5437-92f1-12fccaacc4b3,zendesk,19068095,28779327609372,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4119,ba661159-b0ae-5359-99fb-bcb5d63fdd30,zendesk,19068207,28779462412572,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4120,495ee938-074b-585e-96ee-c595f2608968,zendesk,19069950,28782152944796,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775


In [60]:
# drop not unique ticket uuids
llm_ticket_insert_final = llm_ticket_insert_final.drop_duplicates(subset=["ticket_uuid"], keep='first')

In [61]:
llm_ticket_insert_final.shape

(4071, 9)

In [73]:
# llm attachments
llm_attachment_insert["ticket_uuid"] = llm_attachment_insert.apply(
    lambda row: generate_ticket_uuid(
        row.get("zendesk_id"), row.get("comment_id"), None
    ),
    axis=1,
)
llm_attachment_insert["status"] = "new"

def extract_file_name_and_extension(url):
    """
    Extract the file name and extension from a URL.
    Returns a tuple (file_name, file_extension).
    """
    base_name = os.path.basename(url)
    file_name, file_extension = os.path.splitext(base_name)
    file_name = file_name.replace("?name=", "")  # Replace spaces with underscores
    extension = file_extension.lstrip('.')
    return file_name + "." + extension , extension  # Remove leading dot from extension

llm_attachment_insert[['file_name', 'file_extension']] = llm_attachment_insert['attachment_url'].apply(
    lambda x: pd.Series(extract_file_name_and_extension(x))
)

In [79]:
llm_attachment_insert['created_at'] = llm_attachment_insert.merge(
    llm_ticket_insert_final[['ticket_uuid', 'created_at']],
    left_on='ticket_uuid',
    right_on='ticket_uuid',
    how='left'
)['created_at_y']
llm_attachment_insert['status_written_at'] = llm_attachment_insert['created_at']

In [81]:
def get_s3_key(attachment_id, url, s3_object_key_base):
    """
    Generate the S3 object key for an attachment based on its ID and URL.
    """
    ext = os.path.splitext(url.split("name=")[-1])[-1] if "name=" in url else ".pdf"
    object_key = os.path.join(s3_object_key_base, f"{attachment_id}{ext}")
    return object_key

llm_attachment_insert['s3_key'] = llm_attachment_insert.apply(
    lambda row: get_s3_key(row['attachment_id'], row['attachment_url'], "ocr_source_files/letters"),
    axis=1
)

llm_attachment_insert['s3_bucket'] = "pair-email-classification"

In [82]:
llm_attachment_insert_final = llm_attachment_insert[["ticket_uuid", "attachment_id", "zendesk_id", "comment_id", "status", "file_name", "file_extension", "status_written_at", "created_at", "s3_key", "s3_bucket"]]

In [83]:
llm_attachment_insert_final

,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,s3_key,s3_bucket
0,75fc9480-1a6d-52b1-a8ba-d861655f9b27,28787575304348-1,19072799,28787575304348,new,Dokument_197426_09072026_132935.pdf,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28787575304348-1.pdf,pair-email-classification
1,40033f7e-9347-5c15-a9fc-a29db526a08d,28787295872668-1,19072672,28787295872668,new,5636fb4cb87e48b420260709-182-13ya2t7.pdf,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28787295872668-1.pdf,pair-email-classification
2,8a7ef91a-394f-5d13-987c-3e3cd8cbcb38,28787282083740-1,19072671,28787282083740,new,DR_II_2706_26_An_Auftraggeber_Weiterleitung_Pf...,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28787282083740-1.pdf,pair-email-classification
3,5a9dfa43-6bfa-50e1-a790-7a33647405f2,28787275402652-1,19072667,28787275402652,new,d1ca2aa791d948f220260709-182-15s0kl9.pdf,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28787275402652-1.pdf,pair-email-classification
4,ce30ef5a-b0f1-518e-a796-214519da89bf,28787280433564-1,19072659,28787280433564,new,DR-II_052126_Nachr_Nichtermittlung_08_07_20262...,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28787280433564-1.pdf,pair-email-classification
...,...,...,...,...,...,...,...,...,...,...,...
4117,f33eebda-16a8-51f7-839f-d8482f3e93ad,28779319216284-1,19068087,28779319216284,new,combined_pdf_1_Sammel1_1_Sammel220260709-183-j...,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28779319216284-1.pdf,pair-email-classification
4118,694e0ba9-08c4-5437-92f1-12fccaacc4b3,28779327609372-1,19068095,28779327609372,new,Dokumente_160626_07072026_17313020260709-182-v...,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28779327609372-1.pdf,pair-email-classification
4119,ba661159-b0ae-5359-99fb-bcb5d63fdd30,28779462412572-1,19068207,28779462412572,new,DR_II_1250_26_aus_Schreibmaschine20260709-183-...,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28779462412572-1.pdf,pair-email-classification
4120,495ee938-074b-585e-96ee-c595f2608968,28782152944796-1,19069950,28782152944796,new,DR_II_997_26_aus_Schreibmaschine20260709-180-1...,pdf,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775,ocr_source_files/letters/28782152944796-1.pdf,pair-email-classification


In [84]:
llm_ticket_insert_final

,ticket_uuid,source_type,zendesk_id,comment_id,egvp_id,status,origin,created_at,status_written_at
0,75fc9480-1a6d-52b1-a8ba-d861655f9b27,zendesk,19072799,28787575304348,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
1,40033f7e-9347-5c15-a9fc-a29db526a08d,zendesk,19072672,28787295872668,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
2,8a7ef91a-394f-5d13-987c-3e3cd8cbcb38,zendesk,19072671,28787282083740,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
3,5a9dfa43-6bfa-50e1-a790-7a33647405f2,zendesk,19072667,28787275402652,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4,ce30ef5a-b0f1-518e-a796-214519da89bf,zendesk,19072659,28787280433564,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
...,...,...,...,...,...,...,...,...,...
4117,f33eebda-16a8-51f7-839f-d8482f3e93ad,zendesk,19068087,28779319216284,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4118,694e0ba9-08c4-5437-92f1-12fccaacc4b3,zendesk,19068095,28779327609372,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4119,ba661159-b0ae-5359-99fb-bcb5d63fdd30,zendesk,19068207,28779462412572,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775
4120,495ee938-074b-585e-96ee-c595f2608968,zendesk,19069950,28782152944796,None,new,DE,2026-07-15 12:04:34.164775,2026-07-15 12:04:34.164775


In [88]:
import pandas as pd

def categorical_summary(df, cols=None):
    if cols is None:
        cols = df.select_dtypes(include=['object', 'category']).columns

    summary = pd.DataFrame({
        'dtype': df[cols].dtypes,
        'n_unique': df[cols].nunique(),
        'n_missing': df[cols].isna().sum(),
        'pct_missing': (df[cols].isna().mean() * 100).round(2),
        'most_frequent': df[cols].mode().iloc[0],
        'freq_most_frequent': df[cols].apply(lambda x: x.value_counts().iloc[0] if not x.value_counts().empty else 0),
    })
    return summary.sort_values('n_missing', ascending=False)

categorical_summary(llm_ticket_insert_final)

,dtype,n_unique,n_missing,pct_missing,most_frequent,freq_most_frequent
egvp_id,object,0,4071,100.0,NaN,0
ticket_uuid,object,4071,0,0.0,0027abcd-cae0-5d1e-8b9e-5334d34ebd5d,1
source_type,object,1,0,0.0,zendesk,4071
status,object,1,0,0.0,new,4071
origin,object,1,0,0.0,DE,4071


In [89]:
categorical_summary(llm_attachment_insert_final)

,dtype,n_unique,n_missing,pct_missing,most_frequent,freq_most_frequent
ticket_uuid,object,4071,0,0.0,98a3c604-3371-5d6f-b9fb-cc329813811f,5
attachment_id,object,4122,0,0.0,27262867171740-1,1
status,object,1,0,0.0,new,4122
file_name,object,4100,0,0.0,0_Sammel1.pdf,10
file_extension,object,5,0,0.0,pdf,4039
s3_key,object,4122,0,0.0,ocr_source_files/letters/27262867171740-1.pdf,1
s3_bucket,object,1,0,0.0,pair-email-classification,4122


In [85]:
llm_ticket_insert_final.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/letters/courteam_backlog_letters/data/llm_ticket_insert_final.csv", index=False)
llm_attachment_insert_final.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/letters/courteam_backlog_letters/data/llm_attachment_insert_final.csv", index=False)


# below is when i run textract here 

In [ ]:
# set object key and text with page markers
results= pd.DataFrame(results.items(), columns=['s3_object_key', 'text_with_page_markers'])
results['attachment_id'] = results['s3_object_key'].apply(lambda x: os.path.basename(x).split('.')[0])
# merge
final_df = pd.merge(backlog_df_new, results, on='attachment_id', how='left')

In [ ]:
# remove page markers
final_df['text'] = final_df['text_with_page_markers'].str.replace(r'\<page_\d+\>', '', regex=True)

In [ ]:
final_df

In [ ]:
#final_df.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/court_team_backlog/backlog_data_court_team_80_percent.csv", index=False)

In [ ]:
final_df

In [ ]:
final_df.text_with_page_markers.isna().sum()

In [ ]:
final_df.text_with_page_markers.apply(lambda x: len(x) if isinstance(x, str) else 0).describe()

In [ ]:
short = final_df[final_df.text_with_page_markers.apply(lambda x: len(x) if isinstance(x, str) else 0) < 10]
short 

In [ ]:
final_df.shape

In [ ]:
# drop shorts
final_df = final_df[final_df.text_with_page_markers.apply(lambda x: len(x) if isinstance(x, str) else 0) >= 10]


In [ ]:
final_df.shape

In [ ]:
final_df.text_with_page_markers.apply(lambda x: len(x) if isinstance(x, str) else 0).describe()

In [ ]:
final_df.reset_index(drop=True, inplace=True)

In [ ]:
final_df.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/court_team_backlog/backlog_data_court_team_full.csv", index=False)

In [ ]:
final_df

In [ ]:
final_df